In [1]:
%pip install requests beautifulsoup4 pymysql

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [32]:
import requests
from bs4 import BeautifulSoup

url = "http://www.bspider.top/anjuke/"
r = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36 Edg/156.0.0.0"}, timeout=30)
r.raise_for_status()
r.encoding = r.apparent_encoding
soup = BeautifulSoup(r.text, "html.parser")

In [33]:
rows = soup.select("div.property")
print(len(rows))

60


In [34]:
row = rows[0]
node = row.select_one("h3")

In [35]:
title_node = row.select_one("div.property-content-title h3")
if title_node:
    title = title_node.get_text(strip=True)
else:
    title = ""
print("标题title：", title)


标题title： 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器


In [36]:
house_node = row.select_one(" p.property-content-info-comm-name")
house = house_node.get_text(strip=True) if house_node else ""
print("楼盘house：", house)


楼盘house： 万科金域学府翰林菁华里


In [37]:
addr_node = row.select_one(" p.property-content-info-comm-address")
address = addr_node.get_text(strip=True) if addr_node else ""
print("地址address：", address)


地址address： 南岸学府大道二塘路59号


In [1]:
import re
import requests
from bs4 import BeautifulSoup

BASE_URL = "http://www.bspider.top/anjuke/"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36 Edg/156.0.0.0"}

def text_of(node):
    return node.get_text(strip=True) if node else ""


r = requests.get(BASE_URL, headers=HEADERS, timeout=30)
r.raise_for_status()
soup = BeautifulSoup(r.text, "html.parser")


rows = soup.select("div.property")  
row = rows[0]                       
title = text_of(row.select_one(".property-content-title-name")) or text_of(row.select_one("h3"))
house = text_of(row.select_one(".property-content-info-comm-name"))

address_nodes = row.select(".property-content-info-comm-address span")
if address_nodes:
    address = "".join(n.get_text(strip=True) for n in address_nodes)
else:
    address = text_of(row.select_one(".property-content-info-comm-address"))

attr_nodes = row.select(".property-content-detail > section > div:nth-child(1) .property-content-info-text")
struct = area = floor = maketime = ""
if len(attr_nodes) >= 1:
    struct = attr_nodes[0].get_text(strip=True)
if len(attr_nodes) >= 2:
    area = attr_nodes[1].get_text(strip=True)
if len(attr_nodes) >= 3:
    floor = attr_nodes[2].get_text(strip=True)
if len(attr_nodes) >= 4:
    maketime = attr_nodes[3].get_text(strip=True)


price = text_of(row.select_one(".property-price-average"))
if not price:
    m = re.search(r"\d[\d,.]*\s*元\s*/\s*(?:㎡|平米|平方米)", text_of(row.select_one(".property-price")))
    price = m.group(0).replace(" ", "") if m else ""

print("===== 首条房源8字段 =====")
print(f"title    : {title}")
print(f"house    : {house}")
print(f"address  : {address}")
print(f"struct   : {struct}")
print(f"area     : {area}")
print(f"floor    : {floor}")
print(f"maketime : {maketime}")
print(f"price    : {price}")


===== 首条房源8字段 =====
title    : 万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器
house    : 万科金域学府翰林菁华里
address  : 南岸学府大道二塘路59号
struct   : 3室2厅1卫
area     : 89.9㎡
floor    : 南北
maketime : 低层(共27层)
price    : 16465元/㎡


In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin
import csv

base_url = "http://www.bspider.top/anjuke/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/156.0.0.0 Safari/537.36 Edg/156.0.0.0",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "zh-CN,zh;q=0.9",
    "Referer": "http://www.bspider.top/anjuke/",
}


def parse_one_item(item):
   
    # title 标题
    title_node = item.select_one("h3.property-content-title-name")
    title = title_node.get_text(strip=True) if title_node else ""

    # house 楼盘名称
    house_node = item.select_one("p.property-content-info-comm-name")
    house = house_node.get_text(strip=True) if house_node else ""

    # address 地址，拼接多个span
    addr_spans = item.select(".property-content-info-comm-address span")
    address = "".join(s.get_text(strip=True) for s in addr_spans)

    info_ps = item.select("div.property-content-info p.property-content-info-text")

    struct = info_ps[0].get_text(strip=True) if len(info_ps)>=1 else ""   # struct 户型
    area = info_ps[1].get_text(strip=True) if len(info_ps)>=2 else ""    # area 面积
    floor = info_ps[3].get_text(strip=True) if len(info_ps)>=4 else ""   # floor 楼层

    maketime = ""
    for p in info_ps:
        m = re.search(r"(\d{4})年建造", p.get_text(strip=True))
        if m:
            maketime = m.group(1)
            break

    # price 单价
    price_node = item.select_one("p.property-price-average")
    price = price_node.get_text(strip=True) if price_node else ""


    return {
        "title": title,
        "house": house,
        "address": address,
        "struct": struct,
        "area": area,
        "floor": floor,
        "maketime": maketime,
        "price": price
    }


def main():
    current_url = base_url
    page_count = 1
    raw_list = []
    while True:
        print(f"\n=====正在采集第{page_count}页，current_url={current_url}=====")
        try:
            r = requests.get(current_url, headers=headers, timeout=10)
            r.raise_for_status()
        except requests.exceptions.Timeout:
            print("❌ 请求超时，网络无法连通网页，停止采集")
            break
        except Exception as e:
            print(f"❌ 请求异常：{e}")
            break

        r.encoding = r.apparent_encoding
        soup = BeautifulSoup(r.text, "html.parser")
        items = soup.select("div.property")
        if not items:
            print("当前页面无房源，停止采集")
            break

        for item in items:
            house_dict = parse_one_item(item)
            print(house_dict)
            raw_list.append(house_dict)

        # 下一页逻辑保持原样
        next_a = soup.select_one("a.next-active")
        if not next_a:
            print("✅未找到下一页a标签，到达末页，采集终止")
            break
        href_raw = next_a["href"]
        print(f"下一页href原值：{href_raw}")
        next_full_url = urljoin(base_url, href_raw)
        print(f"urljoin补全后的完整地址：{next_full_url}")
        current_url = next_full_url
        page_count += 1

    print("\n====全部页面采集完成====")
    print("✅去重前一共采集房源数量：", len(raw_list))

    seen_key = set()
    unique_house = []
    for d in raw_list:
        # 8字段完全一致才算重复，顺序与实验要求一致
        key = (
            d["title"], d["house"], d["address"], d["struct"],
            d["area"], d["floor"], d["maketime"], d["price"]
        )
        if key not in seen_key:
            seen_key.add(key)
            unique_house.append(d)
    print("✅去重后房源数量：", len(unique_house))

    field_names = ["title", "house", "address", "struct", "area", "floor", "maketime", "price"]
    with open("data/houseinfo.csv", "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=field_names)
        writer.writeheader()
        writer.writerows(unique_house)

    print("✅去重后数据已保存到 data/houseinfo.csv")


if __name__ == "__main__":
    main()



=====正在采集第1页，current_url=http://www.bspider.top/anjuke/=====
{'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019', 'price': '16465元/㎡'}
{'title': '黄泥磅 两代一家 正规三房双卫 业主自住房 拎包入住', 'house': '两代一家', 'address': '渝北渝北黄泥磅紫荆路29号', 'struct': '3室2厅2卫', 'area': '94.5㎡', 'floor': '低层(共28层)', 'maketime': '2008', 'price': '14815元/㎡'}
{'title': '鸳鸯16年次新小区 大雅云居山刚需经典3房户型 随时看房诚售', 'house': '大雅云居山', 'address': '渝北鸳鸯龙展路59号', 'struct': '3室2厅1卫', 'area': '88㎡', 'floor': '低层(共24层)', 'maketime': '2016', 'price': '16487元/㎡'}
{'title': '中交中央公园 横厅精装大两房 出门都是公园', 'house': '中交中央公园悦颂', 'address': '渝北中央公园翠丹路4号', 'struct': '2室2厅1卫', 'area': '91㎡', 'floor': '低层(共25层)', 'maketime': '2016', 'price': '20660元/㎡'}
{'title': '富力现代广场 豪装大两房 户型方正 看房方便', 'house': '富力现代广场', 'address': '南岸弹子石弹子石新街59号', 'struct': '2室2厅1卫', 'area': '85.5㎡', 'floor': '低层(共30层)', 'maketime': '2012', 'price': '16969元/㎡'}
{'title': '简奢风格

In [3]:
import csv

with open("data/houseinfo.csv", "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    read_data = list(reader)

data_count = len(read_data)
print(f"DictReader回读，有效数据行数（表头不计入）：{data_count}")
if read_data:
    print("首条样本：", read_data[0])
    print("最后一条样本：", read_data[-1])


DictReader回读，有效数据行数（表头不计入）：498
首条样本： {'title': '万科物业自营 南坪六公里金域翰林 三号线精装正三房带家具电器', 'house': '万科金域学府翰林菁华里', 'address': '南岸学府大道二塘路59号', 'struct': '3室2厅1卫', 'area': '89.9㎡', 'floor': '低层(共27层)', 'maketime': '2019', 'price': '16465元/㎡'}
最后一条样本： {'title': '恒大照母山 精装3房 内圈位置 户型方正 拎包即住 诚意出售', 'house': '恒大照母山山水城', 'address': '渝北照母山恒康路9号', 'struct': '3室2厅1卫', 'area': '90.2㎡', 'floor': '低层(共32层)', 'maketime': '2015', 'price': '15530元/㎡'}


In [8]:
import csv
import pymysql

def save_mysql(conn, sql, val):
   
    cursor = conn.cursor()
    try:
       
        cursor.executemany(sql, val)
        conn.commit()
        
        print(f"executemany 受影响行数：{cursor.rowcount}")
    except Exception as err:
        
        conn.rollback()
     
        print("数据库异常，已回滚：", err)
    finally:
        cursor.close()
      

def main():
   
    val_list = []
    with open("data/houseinfo.csv", "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            one_tuple = (
                row["title"],
                row["house"],
                row["address"],
                row["struct"],
                row["area"],
                row["floor"],
                row["maketime"],
                row["price"]
            )
            val_list.append(one_tuple)
    print(f"从CSV读取待入库记录总条数：{len(val_list)}")

    params = {
        "host": "mysql",
        "port": 3306,
        "user": "student",
        "password": "123456",
        "database": "smartpark_lab",
        "charset": "utf8mb4",
        "autocommit": False,
        "connect_timeout": 10   
    }

    
    connect = pymysql.connect(**params)

    try:
        
        cur_check = connect.cursor()
        cur_check.execute("SELECT title,house,address,struct,area,floor,maketime,price FROM houseinfo")
        exist_records = set(cur_check.fetchall())

        new_data = []
        for item in val_list:
            if item not in exist_records:
                new_data.append(item)

        cur_check.close()
        print(f"过滤重复后待插入条数：{len(new_data)}")
       
        insert_sql = """
        INSERT INTO houseinfo(title, house, address, struct, area, floor, maketime, price)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
        if len(new_data) > 0:
            save_mysql(connect, insert_sql, new_data)
        else:
            print("全部数据已存在数据库，无需插入")

    finally:
      
        connect.close()
       
        print("数据库连接已关闭")


if __name__ == "__main__":
    main()


从CSV读取待入库记录总条数：498
过滤重复后待插入条数：0
全部数据已存在数据库，无需插入
数据库连接已关闭
